# Example 8.5.1 / 2: Request a Dual Ray

Companion notebook for **8.5.1 Primal and Dual Rays** in the AMPL lab Documentation, section "Request a Dual Ray". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Request a Dual Ray

In this version each product has a committed minimum, written as a constraint `Commit`, and the
commitments need 41.8 of the 40 hours. With the same steel data plus `commit`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                                 # products
param rate {PROD} > 0;                    # tons produced per hour
param avail >= 0;                         # hours available
param profit {PROD};                      # profit per ton
param market {PROD} >= 0;                 # most tons that can be sold
param commit {PROD} >= 0;                 # tons already promised to customers
var Make {p in PROD} >= 0, <= market[p];  # tons produced, up to the market limit
maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];  # total profit
# production hours fit the hours available
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;
# promised tons must be made
subject to Commit {p in PROD}: Make[p] >= commit[p];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["commit"] = {"bands": 3000, "coils": 2000, "plate": 2000}
ampl.param["avail"] = 40

In [5]:
# run.py
ampl.option["presolve"] = 0
ampl.solve(solver="highs", highs_options="alg:rays=2", verbose=False)
print(ampl.solve_result)
print(ampl.get_data("Commit.dunbdd").to_pandas())
print("Time:", ampl.get_value("Time.dunbdd"))

infeasible
       Commit.dunbdd
bands       0.005000
coils       0.007143
plate       0.006250
Time: -1


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
param rate := bands 200  coils 140  plate 160;
param profit := bands 25  coils 30  plate 29;
param market := bands 6000  coils 4000  plate 3500;
param commit := bands 3000  coils 2000  plate 2000;
param avail := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option presolve 0;
option solver highs;
option highs_options 'alg:rays=2';
option solver_msg 0;
solve;
display solve_result, Commit.dunbdd, Time.dunbdd;

HiGHS 1.14.0:   alg:rays = 2

suffix dunbdd OUT;
solve_result = infeasible

Commit.dunbdd [*] :=
bands  0.005
coils  0.00714286
plate  0.00625
;

Time.dunbdd = -1

